# Query HNSW Index

Loads the saved HNSW index, paragraph texts, and metadata without needing the original dataset, then runs nearest-neighbor search for a text query.

In [1]:

import json
from pathlib import Path
import hnswlib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Paths
index_dir = Path("../data/index/wiki_hnsw")
manifest = json.loads((index_dir / "manifest.json").read_text())
meta_path = index_dir / manifest["metadata_file"]
para_path = index_dir / manifest["paragraphs_file"]
index_path = index_dir / manifest["hnsw_index_file"]

# Load model (uses same name as during indexing)
model = SentenceTransformer(manifest["model"])

# Load index
index = hnswlib.Index(space=manifest["metric"], dim=manifest["dim"])
index.load_index(str(index_path))
index.set_ef(manifest.get("ef_search", 64))
print("Index loaded", index_path)
print("Vectors:", index.get_current_count())


/opt/anaconda3/envs/wikiqa/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 7940.31it/s]


Index loaded ../data/index/wiki_hnsw/index.bin
Vectors: 1160


In [2]:

# Check for NaNs in stored embeddings (requires embeddings.dat memmap)
import numpy as np
emb_path = index_dir / manifest.get("embeddings_file", "embeddings.dat")
if emb_path.exists():
    emb = np.memmap(emb_path, dtype="float32", mode="r", shape=(manifest["vectors"], manifest["dim"]))
    nan_mask = np.isnan(emb).any(axis=1)
    num_nan = int(nan_mask.sum())
    print(f"Vectors with NaNs: {num_nan} / {emb.shape[0]}")
else:
    print(f"Embeddings file not found: {emb_path}")


Vectors with NaNs: 0 / 1160


In [3]:

# Load metadata and paragraphs (aligned by vector id)
metadata = []
with meta_path.open() as f:
    for line in f:
        metadata.append(json.loads(line))
paragraphs = []
with para_path.open() as f:
    for line in f:
        paragraphs.append(json.loads(line)["text"])
assert len(metadata) == len(paragraphs) == manifest["vectors"]
print(f"Loaded {len(paragraphs)} paragraphs")


Loaded 1160 paragraphs


In [4]:

def search(query: str, k: int = 5):
    # Encode and normalize; guard against NaNs from zero vectors
    vec = model.encode_query([query], convert_to_numpy=True, normalize_embeddings=False)[0]
    labels, dists = index.knn_query(vec, k=k)
    rows = []
    for idx, dist in zip(labels[0], dists[0]):
        meta = metadata[idx]
        rows.append(
            {
                "score": 1 - dist,
                "title": meta.get("title"),
                "url": meta.get("url"),
                "paragraph": paragraphs[idx],
            }
        )
    return pd.DataFrame(rows)


In [5]:

# Example query
search("earthquake in haiti", k=5)


,score,title,url,paragraph
0,0.651094,List of natural disasters in Haiti,https://en.wikipedia.org/wiki/List_of_natural_...,"1691 \n EarthquakeJules Trousset, Nouveau Dict..."
1,0.635575,List of natural disasters in Haiti,https://en.wikipedia.org/wiki/List_of_natural_...,2009\n20 October: heavy rain in the Haitian ca...
2,0.598598,List of natural disasters in Haiti,https://en.wikipedia.org/wiki/List_of_natural_...,2021\n 4 July: Hurricane Elsa passed near the ...
3,0.554945,List of natural disasters in Haiti,https://en.wikipedia.org/wiki/List_of_natural_...,Haiti's unique position and geography in the C...
4,0.506522,List of natural disasters in Haiti,https://en.wikipedia.org/wiki/List_of_natural_...,2007\n17 March: floods caused by rain and stor...


In [6]:
index

<hnswlib.Index(space='ip', dim=768)>

In [7]:

# Helper to format top search results as context strings
def format_context(df, max_chars=1200):
    chunks = []
    for _, row in df.iterrows():
        para = row['paragraph'] or ''
        chunks.append(f"Title: {row.get('title','')}. URL: {row.get('url','')}. Paragraph: {para}")
    context = "\n".join(chunks)
    return context[:max_chars]


In [9]:

# Load a lightweight Qwen model (<3B) for answering
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

qa_model_name = "Qwen/Qwen2-1.5B-Instruct"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForCausalLM.from_pretrained(qa_model_name, torch_dtype="auto", device_map="auto")
qa_pipe = pipeline("text-generation", model=qa_model, tokenizer=qa_tokenizer)


Loading weights: 100%|██████████| 338/338 [00:04<00:00, 72.84it/s] 


In [11]:

# Ask a question using top search hits as RAG context

def answer(question: str, k: int = 5, max_new_tokens: int = 128):
    results = search(question, k=k)
    context = format_context(results)
    prompt = (f"You are a concise assistant. Use only the provided context to answer.vContext: {context} Question: {question} Answer:")
    out = qa_pipe(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    return results, context, out[0]['generated_text'].split('Answer:',1)[-1].strip()

# Example
res_df, context, answer_text = answer("What natural disasters are listed for Haiti?", k=5)
res_df, context, answer_text


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


(      score                               title  \
 0  0.718486  List of natural disasters in Haiti   
 1  0.690331  List of natural disasters in Haiti   
 2  0.683868  List of natural disasters in Haiti   
 3  0.681307  List of natural disasters in Haiti   
 4  0.662340  List of natural disasters in Haiti   
 
                                                  url  \
 0  https://en.wikipedia.org/wiki/List_of_natural_...   
 1  https://en.wikipedia.org/wiki/List_of_natural_...   
 2  https://en.wikipedia.org/wiki/List_of_natural_...   
 3  https://en.wikipedia.org/wiki/List_of_natural_...   
 4  https://en.wikipedia.org/wiki/List_of_natural_...   
 
                                            paragraph  
 0  The following is a non-exhaustive list of natu...  
 1  1691 \n EarthquakeJules Trousset, Nouveau Dict...  
 2  Haiti's unique position and geography in the C...  
 3                     List\nHaiti\nNatural disasters  
 4  2009\n20 October: heavy rain in the Haitian ca...  ,
 'Tit